# Homework 4 — Clothing Q-learning desk (Module 4 framework)

**Salih Tutun, PhD** · DAT 5668 · Module 4

**Individual.** Due after **Session 8**.

Open this file in **Google Colab** from Canvas (or run it locally). When you are done, download the `.ipynb` and upload it on the Canvas assignment.

Session 8 practiced tabular Q-learning on **Electronics**. This homework is the **same framework** on a **new business desk: Clothing** — so you cannot copy class numbers. You DEFINE the MDP pieces, write the decide + update functions, design a reward table, break it on purpose, and leave a framework line a manager can read.

| Session skill | What you reuse here |
| --- | --- |
| 7 | Hand Q-update idea: $Q(s,a) \leftarrow Q(s,a) + \alpha(r + \gamma \max_{a'} Q(s',a') - Q(s,a))$ |
| 8 | `epsilon_greedy` (= decide) + `q_update` on real inventory days |
| Framework | DEFINE → PIPELINE → interpret → human override |

**Deliverable:** every `<<< FILL` completed and `review_hw4()` prints **PASS**.

---

## How to run this in Colab

1. Open from Canvas **Homework 4 — Open in Colab**, or use [this Colab file](https://colab.research.google.com/github/salihtutun/Agentic-AI_Autonomous_Decision_Making/blob/main/Students/Homework/HW4-Clothing-Q-Learning.ipynb).
2. Run cells **top to bottom** (Shift+Enter). Do not skip the data cell.
3. Fill Parts 1–6. Re-run a cell after you edit it.
4. Run `review_hw4()` until it prints **PASS**.
5. File → Download → `.ipynb`. Upload on **Homework 4 — Clothing Q-learning**.

If a cell says `NameError`, you skipped a cell above it. Run from the top.

## What “great” looks like

1. DEFINE names Clothing state / actions / reward in business words (not Electronics).
2. `epsilon_greedy` and `q_update` match the Module 4 update picture.
3. Your reward table makes Low→Order and High→Reduce win after training.
4. The broken-reward experiment shows you can **read** a bad policy.
5. Framework line names the family (**RL**) and a **human override**.


## Business story — Clothing buyer desk (new process)

You support a **Clothing** category buyer (fashion / apparel SKUs), not Electronics.

Each day the desk observes inventory relative to typical Clothing demand and chooses:

- **Order** — pull more units onto the shelf  
- **Maintain** — leave the plan alone  
- **Reduce** — cut / mark-down / stop inbound  

State bins (same rule as Session 8, new product):

| State | Meaning |
| --- | --- |
| Low Inventory | inventory / avg demand $< 0.5$ |
| Medium Inventory | between $0.5$ and $2.0$ |
| High Inventory | inventory / avg demand $> 2.0$ |

Data file: `inventory_dataset_full.csv`, product filter **`Clothing`**.

```text
load Clothing days
  → observe state s
  → decide a with ε-greedy on Q
  → reward r from your table
  → next state s' = tomorrow's Clothing state
  → q_update
  → repeat
```

A chatbot that never updates $Q$ is **not** doing Module 4.


## Part 1 — DEFINE (before code)

Fill the next cell for the **Clothing** desk. Do not paste Electronics wording.

| Piece | Your job |
| --- | --- |
| State $s$ | What inventory signal do you observe before acting? |
| Actions $\mathcal{A}$ | List Order, Maintain, Reduce |
| Reward $r$ | What business pain/gain does the table score? |
| Next state $s'$ | Where does tomorrow's Clothing state come from? |
| Not in $s$ | Name one document/exception a person still owns |


In [ ]:
# ============================================================
# PART 1 — FILL (DEFINE in words)
# ============================================================

state_s = ""  # <<< FILL: Clothing inventory bin before the action
actions = []  # <<< FILL: ["Order", "Maintain", "Reduce"]
reward_r = ""  # <<< FILL: stockout / holding / markdown language is fine
next_state = ""  # <<< FILL: next Clothing day state from the dataset
not_in_state = ""  # <<< FILL: e.g. vendor email, photo QA, brand memo

print("s:", state_s)
print("A:", actions)
print("r:", reward_r)
print("s':", next_state)
print("not in s:", not_in_state)


## Part 2 — LOAD (RUN WITH ME)

This cell loads **Clothing** days and builds states. Do not edit the loader unless a path fails — then check Drive `MyDrive/DAT5668/data`.


In [ ]:
# ============================================================
# PART 2 — RUN WITH ME (Clothing data)
# ============================================================

from pathlib import Path
from collections import Counter
import csv
import random
import statistics

DRIVE_DATA_FOLDER = "/content/drive/MyDrive/DAT5668/data"
PRODUCT = "Clothing"  # NEW desk — not Electronics
RNG = random.Random(7)


def course_data(name: str) -> Path:
    """Find course CSV on Drive, local assets, or GitHub raw fallback."""
    candidates = []
    if DRIVE_DATA_FOLDER:
        candidates.append(Path(DRIVE_DATA_FOLDER) / name)
    for base in (Path("data"), Path("../../assets/data"), Path("assets/data"), Path("/content/data")):
        candidates.append(base / name)
    for path in candidates:
        if path.exists():
            return path
    import urllib.request

    dest = Path("/content/data") if Path("/content").exists() else Path("data")
    dest.mkdir(parents=True, exist_ok=True)
    out = dest / name
    url = (
        "https://raw.githubusercontent.com/salihtutun/"
        "Agentic-AI_Autonomous_Decision_Making/main/assets/data/" + name
    )
    urllib.request.urlretrieve(url, out)
    return out


ROWS = []
with course_data("inventory_dataset_full.csv").open(encoding="utf-8") as handle:
    for row in csv.DictReader(handle):
        if (row.get("Product") or "").strip() != PRODUCT:
            continue
        ROWS.append(
            {
                "Current_Inventory": int(float(row["Current_Inventory"])),
                "Demand": int(float(row["Demand"])),
                "Action_Taken": (row.get("Action_Taken") or "").strip(),
            }
        )

AVG_DEMAND = statistics.mean(r["Demand"] for r in ROWS)


def categorize(inv: int) -> str:
    ratio = inv / max(AVG_DEMAND, 1e-9)
    if ratio < 0.5:
        return "Low Inventory"
    if ratio <= 2.0:
        return "Medium Inventory"
    return "High Inventory"


for r in ROWS:
    r["State"] = categorize(r["Current_Inventory"])

STATES = ["Low Inventory", "Medium Inventory", "High Inventory"]
ACTIONS = ["Order", "Maintain", "Reduce"]
N_DAYS = len(ROWS)

print(PRODUCT, "days:", N_DAYS, "| avg demand:", round(AVG_DEMAND, 1))
print("state mix:", dict(Counter(r["State"] for r in ROWS)))
print("Data ready. Next: design rewards, then write ε-greedy + Q-update.")


## Part 3 — Design the Clothing reward table (harder than Session 8)

In class the Electronics table was given. Here **you** set the four critical cells so a Clothing buyer would agree:

| Constraint (must hold) | Why |
| --- | --- |
| `R(Low, Order) > R(Low, Reduce)` | Do not starve a low shelf |
| `R(High, Reduce) > R(High, Order)` | Do not overstock a high shelf |
| Medium: `Maintain` should be best (or tied for best) | Stable fashion plan |

Fill every entry. Then write `reward_rationale` (≥ 60 characters) explaining your Low and High choices in business words.


In [ ]:
# ============================================================
# PART 3 — FILL reward table + rationale
# ============================================================

# Tip: start from Session 8 intuition, then adjust numbers if you want.
# Checker only enforces the inequalities above + Medium Maintain dominance.

REWARDS = {
    ("Low Inventory", "Order"): None,      # <<< FILL number (should be highest in Low row)
    ("Low Inventory", "Maintain"): None,   # <<< FILL
    ("Low Inventory", "Reduce"): None,     # <<< FILL (must be worse than Order)
    ("Medium Inventory", "Order"): None,   # <<< FILL
    ("Medium Inventory", "Maintain"): None,  # <<< FILL (best or tied in Medium)
    ("Medium Inventory", "Reduce"): None,  # <<< FILL
    ("High Inventory", "Order"): None,     # <<< FILL (must be worse than Reduce)
    ("High Inventory", "Maintain"): None,  # <<< FILL
    ("High Inventory", "Reduce"): None,    # <<< FILL (should be highest in High row)
}

reward_rationale = ""  # <<< FILL ≥ 60 characters


def _row_ok():
    for k, v in REWARDS.items():
        if v is None:
            return False, f"missing reward for {k}"
    if not (REWARDS[("Low Inventory", "Order")] > REWARDS[("Low Inventory", "Reduce")]):
        return False, "Low: Order must beat Reduce"
    if not (REWARDS[("High Inventory", "Reduce")] > REWARDS[("High Inventory", "Order")]):
        return False, "High: Reduce must beat Order"
    med = [REWARDS[("Medium Inventory", a)] for a in ACTIONS]
    if REWARDS[("Medium Inventory", "Maintain")] < max(med):
        return False, "Medium: Maintain should be best (or tied)"
    if len(reward_rationale.strip()) < 60:
        return False, "reward_rationale needs ≥ 60 characters"
    return True, "reward table OK"


ok, msg = _row_ok()
print(msg)
for s in STATES:
    print(s, {a: REWARDS[(s, a)] for a in ACTIONS})


## Part 4 — PIPELINE: write `epsilon_greedy` + `q_update`

Same functions as Session 8 Stop 4. Complete them, then train on Clothing days.

$$
Q(s,a) \leftarrow Q(s,a) + \alpha \big(r + \gamma \max_{a'} Q(s',a') - Q(s,a)\big)
$$


In [ ]:
# ============================================================
# PART 4 — FILL ε-greedy + Q-update, then train
# ============================================================

alpha = 0.3
epsilon = 0.5
gamma = 0.7
n_episodes = 50

Q = {s: {a: 0.0 for a in ACTIONS} for s in STATES}


def epsilon_greedy(state, Q, epsilon):
    """Decide: explore random action with prob epsilon, else greedy."""
    # <<< FILL
    return None


def q_update(Q, state, action, reward, next_state, alpha, gamma):
    """Update Q[state][action] in place with the TD target."""
    # <<< FILL
    return None


def train_q(rewards, seed=7):
    """Run the Module 4 pipeline on Clothing consecutive days."""
    local_rng = random.Random(seed)
    Q_local = {s: {a: 0.0 for a in ACTIONS} for s in STATES}
    for _ in range(n_episodes):
        for i in range(len(ROWS) - 1):
            state = ROWS[i]["State"]
            # inline ε-greedy so train_q stays testable even if student RNG differs
            if local_rng.random() < epsilon:
                action = local_rng.choice(ACTIONS)
            else:
                action = max(Q_local[state], key=Q_local[state].get)
            reward = rewards[(state, action)]
            next_state = ROWS[i + 1]["State"]
            old = Q_local[state][action]
            Q_local[state][action] = old + alpha * (
                reward + gamma * max(Q_local[next_state].values()) - old
            )
    return Q_local


# --- Student functions must work on a tiny hand check ---
_hand_Q = {"Low Inventory": {"Order": 0.0, "Maintain": 0.0, "Reduce": 0.0}}
try:
    a0 = epsilon_greedy("Low Inventory", _hand_Q, 0.0)
    q_update(_hand_Q, "Low Inventory", "Order", 2.0, "Low Inventory", 0.5, 0.9)
    hand_ok = a0 == "Order" and abs(_hand_Q["Low Inventory"]["Order"] - 1.0) < 1e-9
except Exception as err:
    hand_ok = False
    print("NOT YET — fix epsilon_greedy / q_update:", err)

if hand_ok:
    Q = train_q(REWARDS)
    policy = {s: max(Q[s], key=Q[s].get) for s in STATES}
    agree = sum(1 for r in ROWS if policy[r["State"]] == r["Action_Taken"])
    print("Learned policy:", policy)
    print("Agree with historical Action_Taken:", agree, "/", N_DAYS, f"({agree / N_DAYS:.1%})")
    print(
        "CHECK 4 —",
        "PASS"
        if policy["Low Inventory"] == "Order" and policy["High Inventory"] == "Reduce"
        else "LOOK — expected Low→Order and High→Reduce with a sensible reward table",
    )
else:
    policy = None
    print("Fill epsilon_greedy and q_update, then re-run this cell.")


## Part 5 — Break the reward on purpose (interpret the policy)

Copy your table into `BROKEN`, then **flip** the Low row so Reduce looks best (and Order looks bad). Retrain. Compare policies.

Write `NOTE` (≥ 100 characters): what did the greedy Clothing policy do under the broken table, and why would a buyer refuse to deploy it?


In [ ]:
# ============================================================
# PART 5 — FILL broken rewards + NOTE
# ============================================================

BROKEN = dict(REWARDS)  # start from your table
# <<< FILL: make Low→Reduce look best (example direction: Reduce = +5, Order = -1)
BROKEN[("Low Inventory", "Reduce")] = None
BROKEN[("Low Inventory", "Order")] = None

NOTE = ""  # <<< FILL ≥ 100 characters


def _broken_ready():
    if None in (BROKEN[("Low Inventory", "Reduce")], BROKEN[("Low Inventory", "Order")]):
        return False, "Set both BROKEN Low Reduce and Low Order numbers"
    if not (
        BROKEN[("Low Inventory", "Reduce")] > BROKEN[("Low Inventory", "Order")]
    ):
        return False, "Broken table must prefer Reduce over Order when Low"
    if len(NOTE.strip()) < 100:
        return False, "NOTE needs ≥ 100 characters"
    return True, "broken setup OK"


bok, bmsg = _broken_ready()
print(bmsg)
if bok and hand_ok:
    Q_bad = train_q(BROKEN, seed=7)
    policy_bad = {s: max(Q_bad[s], key=Q_bad[s].get) for s in STATES}
    print("Policy with YOUR rewards:", policy)
    print("Policy with BROKEN rewards:", policy_bad)
    print(
        "CHECK 5 —",
        "PASS" if policy_bad["Low Inventory"] == "Reduce" else "LOOK — broken Low should learn Reduce",
    )
else:
    policy_bad = None


## Part 6 — Framework line (Module 4)

Name the family and one sentence you will keep. Also name a **human override** if a Q-row ever says Reduce on Low Inventory during a live week.


In [ ]:
# ============================================================
# PART 6 — FILL framework
# ============================================================

framework_family = ""  # <<< FILL: LLM / RL / hybrid  (Clothing Q-table → RL or hybrid)
framework_line = ""    # <<< FILL ≥ 40 characters
human_override = ""    # <<< FILL ≥ 40 characters — who stops a bad Reduce-on-Low?

print("Family:", framework_family or "(FILL)")
print("I will keep:", framework_line or "(FILL)")
print("Human override:", human_override or "(FILL)")


In [ ]:
# ============================================================
# REVIEW — run last. Must print PASS
# ============================================================

def review_hw4():
    """Student-facing checker for Homework 4."""
    issues = []

    # Part 1
    if len((state_s or "").strip()) < 12:
        issues.append("Part 1: state_s — describe the Clothing inventory state.")
    if set(a.lower() for a in (actions or [])) < {"order", "maintain", "reduce"}:
        issues.append('Part 1: actions must include Order, Maintain, Reduce.')
    if len((reward_r or "").strip()) < 10:
        issues.append("Part 1: reward_r — say what the table scores.")
    if len((next_state or "").strip()) < 10:
        issues.append("Part 1: next_state — tomorrow's Clothing state.")
    if len((not_in_state or "").strip()) < 8:
        issues.append("Part 1: not_in_state — name a document/exception path.")

    # Part 3
    ok3, msg3 = _row_ok()
    if not ok3:
        issues.append("Part 3: " + msg3)

    # Part 4 function behavior
    try:
        probe = {s: {a: 0.0 for a in ACTIONS} for s in STATES}
        probe["Low Inventory"]["Order"] = 1.0
        probe["Low Inventory"]["Maintain"] = 0.0
        probe["Low Inventory"]["Reduce"] = 0.0
        greedy = epsilon_greedy("Low Inventory", probe, 0.0)
        if greedy != "Order":
            issues.append("Part 4: epsilon_greedy(..., epsilon=0) must return the argmax action.")
        demo = {"S": {"A": 0.0, "B": 0.0}}
        # use Clothing-shaped dict for q_update
        demoQ = {"Low Inventory": {"Order": 0.0, "Maintain": 0.0, "Reduce": 0.0}}
        q_update(demoQ, "Low Inventory", "Order", 2.0, "Low Inventory", 0.5, 0.0)
        if abs(demoQ["Low Inventory"]["Order"] - 1.0) > 1e-9:
            issues.append("Part 4: q_update math — with γ=0, α=0.5, r=2 from 0 should become 1.0.")
    except Exception as err:
        issues.append(f"Part 4: epsilon_greedy/q_update error — {err}")

    if policy is None:
        issues.append("Part 4: train a policy (Low→Order, High→Reduce) before review.")
    else:
        if policy.get("Low Inventory") != "Order":
            issues.append("Part 4: with your REWARDS, Low Inventory should prefer Order.")
        if policy.get("High Inventory") != "Reduce":
            issues.append("Part 4: with your REWARDS, High Inventory should prefer Reduce.")

    # Part 5
    ok5, msg5 = _broken_ready()
    if not ok5:
        issues.append("Part 5: " + msg5)
    elif policy_bad is not None and policy_bad.get("Low Inventory") != "Reduce":
        issues.append("Part 5: broken table should teach Low→Reduce.")

    # Part 6
    fam = (framework_family or "").strip()
    if fam not in {"LLM", "RL", "hybrid"}:
        issues.append("Part 6: framework_family must be LLM, RL, or hybrid.")
    if fam == "LLM":
        issues.append("Part 6: a Clothing Q-table is numeric RL (or hybrid with a human override) — not pure LLM.")
    if len((framework_line or "").strip()) < 40:
        issues.append("Part 6: framework_line needs ≥ 40 characters.")
    if len((human_override or "").strip()) < 40:
        issues.append("Part 6: human_override needs ≥ 40 characters.")

    if issues:
        print("NOT YET")
        for issue in issues:
            print("-", issue)
        return False

    print("PASS")
    print("Policy:", policy)
    print(fam + ":", framework_line)
    return True


review_hw4()


## Submit checklist

1. [ ] Part 1 DEFINE filled for **Clothing** (not Electronics)  
2. [ ] Part 3 reward inequalities hold + rationale ≥ 60 characters  
3. [ ] Part 4 `epsilon_greedy` + `q_update` → Low→Order, High→Reduce  
4. [ ] Part 5 broken experiment + `NOTE` ≥ 100 characters  
5. [ ] Part 6 framework family + human override  
6. [ ] `review_hw4()` prints **PASS**

**Turn in:** this completed notebook on Canvas (**Homework 4 — Clothing Q-learning**).
